*Your Name*

*Collaborator's Names*

# Asynchronous Operations

So far in this course, we've tried to avoid things like CPU context switching and using threads. With asynchronous operations, we instead want to embrace those things. What matters is the context: so far, we've been working with CPU-bound work. If our problem is I/O-bound, context switching and threads become beneficial. 

## Defining and Awaiting Coroutines

Asynchronous operations rely on two Python keywords: `async` and `await`.

The `async` keyword is used to designate functions that can be run asynchronously; that is, their execution can be paused and resumed by the event loop. Using `async` in a function definition creates a **coroutine function.**

How does the event loop know whether or not a function can be paused and at which point to do it? For that we use the `await` keyword. This tells Python that we need to wait for certain information before we can proceed. Only three types of objects can be awaited: coroutines, tasks, and "futures" (which is a generic category that coroutines and tasks belong to).

Note that the `asyncio` library does *not* need to be imported for the `async` and `await` keywords to be defined.

Now imagine a scenario in which we want to download data from an internet database and save it to file. Downloading the data and opening the file are both operations where we must wait for some external, non-Python entity (the remote database or the local operating system) to finish working before we can proceed. While we're waiting for the remote database to respond (a non-blocking operation in this example), we may as well start opening our file (a blocking operation).
Below we use `async` to define two coroutine functions that mimic downloading data and opening a file.

For the first function, called `fetch_remote_data`, we'll `await` the use of `asyncio.sleep()` to mimic a non-blocking call to a website.
You could replace `asyncio.sleep()` with something like the `requests` library to do actual work.
The coroutine function can be suspended at this line while waiting for the website to respond, returning control to Python so it may do something else (like launch another function) in the meantime.

Our other function is `open_local_file` and uses `time.sleep()` to mimic a call to the operating system. This call is blocking, so there isn't anything to `await` inside the function; however, since we want to interleave the execution of this function with `fetch_remote_data` we must still use `async`. This means we can use `await` with `fetch_remote_data()` itself, as we'll see below.

In [1]:
import asyncio
import time

async def fetch_remote_data(url):  # coroutine function
    print(f"(1) Fetching data from {url}...")
    await asyncio.sleep(2)  # Simulate downloading data (non-blocking)
    print(f"(1) Retrieved data from {url}")
    return f"Data from {url}"

async def open_local_file(file):
    print(f"(2) Loading {file}...")
    time.sleep(1) # Simulate file I/O (blocking)
    print(f"(2) Finished opening {file}")

    return f"{file} open for writing"


---
### Exercises

1. While `fetch_remote_data` is a coroutine *function*, if we "invoke" this function by adding parentheses (and an argument) we get a coroutine *object*. Try running `fetch_remote_data("example.org")` and see what happens. How is this different from a normal (non-coroutine) function?

*Answer here*

2. At the beginning of this section I said that the `await` keyword can only be used with coroutine objects, tasks, and other generic "futures." In that case, what kind of object `asyncio.sleep()` be returning? You can just invoke `asyncio.sleep()` or use the `type()` function to help you answer this question.

*Answer here*

3. Since our `async` functions return coroutine objects when we invoke them with parentheses, we can use `await` on them:

    ```python
    data = await fetch_remote_data("example.org")
    open_file = await open_local_file("local_database.db")

    print(f"\nResults: {data}, {open_file}")  # add newline for nicer printing
    ```

    What happens when we do this? Does `open_local_file` run while `fetch_remote_data` is running? Or do they run one after another like normal functions?

*Answer here*

---
## Awaiting Tasks

At the end of the previous section, you hopefully noticed that `fetch_remote_data` and `open_local_file` ran sequentially, one after the other. This **isn't** the behavior we want; remember, we want to start opening our local file while waiting for data to download over the internet.

While the `async` keyword designates asynchronous functions that can be paused and resumed, coroutines themselves don't actually contain enough information for this to actually *happen*. Instead, we must turn our coroutines into **tasks**. Tasks are objects with additional functionality that allow the event loop to suspend and resume the coroutine contained within.

---
### Exercises

1. The simplest way to create a task is to use `asyncio.create_task`. This returns as task object that can then be awaited:

    ```python
    fetch_task = asyncio.create_task(fetch_remote_data("example.org"))  # create task
    result = await fetch_task  # wait for task to complete

    print(f"\nResult: {result}")  # add newline for nicer printing
    ```

    Run this code and explain the difference between `fetch_task` and `result`. What information do these objects contain? You may find the `type()` function useful for understanding these objects.

*Answer here*

2. Create tasks for `fetch_remote_data("example.org")` and `open_local_file("local_database.db")` and then await their results like in the previous exercise. Do the tasks execute sequentially or do they overlap like we want?

*Answer here*

3. While `asyncio.create_task()` creates tasks that have to be awaited one by one, the `asyncio` library contains other tools for awaiting multiple tasks. The first we'll explore is `asyncio.gather()`, which can take a set of coroutines, turn them into tasks, and wait for the event loop to finish running all of them (we can also give `gather` a set of tasks with the same outcome).

    Run the following code:

    ```python
    # Create coroutine objects
    fetch_coro = fetch_remote_data("example.org")  # coroutine object, not task
    open_coro = open_local_file("local_database.db")
    my_coroutines = [fetch_coro, open_coro]

    # asyncio.gather creates tasks automatically
    # Notice the use of * to unpack the list
    results = await asyncio.gather(*my_coroutines)
    print(f"\nBoth tasks have completed now: {results[0]}, {results[1]}")
    ```

    Do the tasks run in the order you expect?

*Answer here*

4. There's a *third* way we can accomplish this same goal: we can add individual tasks to a `TaskGroup`. This object is used with the `with` context manager. We also have to designate that this block of code is asynchronous using `async`. The advantage is that every task in the group will automatically be awaited:

    ```python
    async with asyncio.TaskGroup() as tg:  # waits for all tasks in group to finish
        task1 = tg.create_task(fetch_remote_data("example.org"))
        task2 = tg.create_task(open_local_file("local_database.db"))
    print(f"\nBoth tasks have completed now: {task1.result()}, {task2.result()}")
    ```

    What's different about how we access the results when using a `TaskGroup` as opposed to `gather`?

*Answer here*

5. You might be wondering what the difference is between `gather` and `TaskGroup`; why would you want to use one instead of the other? The answer comes down to how they handle *failure*.

    **Repeat the `gather` and `TaskGroup` exercises from above but swap `open_local_file` for `simulate_open_failure` below.** Functions that open files for writing often fail if the file exists to prevent us from accidentally overwriting data; this behavior is what we are simulating:

    ```python
    async def simulate_open_failure(file):
        print(f"(2) Loading {file}...")
        raise FileExistsError
    ```

    How do `gather` and `TaskGroup` behave when one of their tasks fails? Ignore the error messages and focus on what the functions are printing (e.g. the lines with `(1)` and `(2)`).

*Answer here*

---
## Using Threads for Blocking Tasks

As a general rule, I/O operations like opening files are blocking. This means the event loop cannot start another task until the I/O is complete. This is the same behavior as if we didn't have asynchronous operations at all.

Thankfully, we can utilize **threads** to recover the benefits of asynchronous operations. So far in this course, we have elected to use processes instead of threads because of Python's Global Interpreter Lock (GIL). Since only one thread can hold the GIL at a time (much like the file locks we used earlier), threads can't usually perform computations in parallel.

Operations like I/O are an exception: since the wait time is driven by the operating system and not Python itself, the GIL doesn't interfere with our work. We can let one thread open a file while another thread proceeds with CPU-bound work. This is particularly useful if we need to open and process several (ideally large) files.

Using threads for asynchronous I/O provides another benefit. When using multiple processes, we should only use one thread per core. But if we're using threads to wait for the operating system, **we can use more threads than we have cores.** The default number of threads that `asyncio` will launch is 20.

Below there are two example functions that mimic opening and processing a text file. Once again we use `time.sleep()` to mimic work being done. Note that `process_text` calls `open_text`; this will be helpful when we introduce threading in the exercises.

In [15]:
def open_text(filename):
    print(f"Opening {filename}...")
    start = time.perf_counter()
    time.sleep(1)  # Simulate file I/O (blocking)
    print(f"Finished opening {filename}; took {time.perf_counter() - start:.2f} seconds")
    return True


async def process_text(filename):

    open_flag = open_text(filename)

    if open_flag:
        print(f"\tProcessing text from {filename}...")
        start = time.perf_counter()
        time.sleep(3)  # Simulate a CPU-bound process
        print(f"\tFinished processing {filename}; took {time.perf_counter() - start:.2f} seconds")


# Our list of sample filenames to "process"
filenames = [f"sample{i}.txt" for i in range(1, 4)]


---
### Exercises

1. Using any of the three methods covered in the previous section, create tasks for and await the results of `process_text()` for each of our three sample files. Use `time.perf_counter()` to time how long the entire process takes.

**Note:** we can't use `%%timeit` effectively with asynchronous operations because of how the event loop executes in Jupyter Notebooks.

In [17]:
# Code here

2. Consider the total time and the time reported by each individual step for the previous question. Are any operations executed in parallel? Which operations are sequential? Why do you think this is?

*Answer here*

3. Rewrite `process_text()` but replace `open_flag = open_text(filename)` with `open_flag = await asyncio.to_thread(open_text, filename)`. Re-time how long it takes to process our three sample files using this new function.

**Note:** as the name implies, `to_thread()` sends a function and its arguments to a new thread. Only standard, synchronous functions can be used with `to_thread()` and `to_thread()` returns a coroutine.

In [18]:
# Code here

4. Again looking at the different timing information, what operations are executed in parallel? Which are sequential? Why do you think this is?

*Answer here*

5. Why wouldn't we want to run `process_text()` in a thread? Recall that `process_text()` is simulating CPU-bound work.

*Answer here*

---
## AsyncIO in Scripts

In Jupyter Notebooks, the event loop is essentially always running. This isn't true in Python scripts. There, we must explicitly start the event loop. The following exercises will walk you through converting code from this Jupyter Notebook into a script.

---
### Exercises

1. Copy and paste your code from the previous example into a script called `asyncio_script.py`. This includes `open_text()`, your *threaded* version of `process_text()`, the list of filenames, and the code you used for creating your various tasks. Don't forget to import `asyncio` and `time`!

2. Try running `asyncio_script.py` without any modification. Depending on the editor you use for your Python scripts, you may already see warning about potential issues. What error do you get?

*Answer here*

3. The above error tells us that we need to put our task code into its own **asynchronous** function (i.e., use `async def`). We might as well put the `filenames` list comprehension and our performance measurements in that same function. This function does not need to take any arguments, as all the information needed should be defined inside the function.

    The traditional choice for such a function is `main()`; that is, the function that executes the "main" body of the script. You can call it whatever you like; `process_all` and `do_work` are reasonable alternatives, though your code will be more easily understood by others if you stick with `main()`.

4. Now that we've encapsulated all of our work into one function (e.g. `main()`), it's time to tell the event loop to run it using `asyncio.run(main())`. Note that we've never needed to use `asyncio.run()` inside this Jupyter Notebook because the event loop is already running!

    Since Python file can also be used as modules (e.g., you can run `from asyncio_script.py import process_text` in some other script), we should only run the event loop when `asyncio_script.py` is being run *as* a script and not as a module. To do that, we check a special hidden variable called `__name__`:

    ```python
    if __name__ == "__main__":
        asyncio.run(main())  # execute main function
    ```

    Go ahead and put this block at the end of `asyncio_script.py`.

5. Run `ascynio_script.py`. Does the output match what you expect from the previous section?

*Answer here*

6. Add `asyncio_script.py` to your git repository and submit it along with this notebook.

## Final Notes

You can combine `asyncio` with `multiprocessing`! In fact, since processes may each have multiple threads, you can even use `asyncio.to_thread()` with `multiprocessing`.

Let's think about the example where we open text files and then process them. If we wanted to combine `asyncio` threading with `multiprocessing`, we could do this in two ways. First, we could use `asyncio` to open all of our images concurrently and then distribute them to a process `Pool`. Second, we could have each process in the `Pool` use `asyncio` to open a subset of files and then process them. Which is better? That really depends on the details! How large are the files you're opening? What is the delay (latency) in opening a file? How complex is the analysis you're doing? The only way to answer this is to do some profiling.

Note that the HPCC has a variety of file systems available. In our lesson on Locks, we discussed that the node's local file system is going to be faster than any of the networked file systems like your Home directory or Scratch. The speed of the filesystem (as well as the speed of your CPU) may affect your results.

## For Next Class

There are few introductory materials I want you to review for next class where we'll learn about two packages called Dask and Modin. These packages are intended to easily parallelize codes that use large NumPy arrays and/or Pandas DataFrames. The readings will focus on Dask, as Modin is easy to explain if you understand what Dask is doing.
If the reading doesn't all make sense immediately, don't worry; I'll be discussing them in more detail in class.

- Watch this [video overview of Dask](https://www.youtube.com/watch?v=nnndxbr_Xq4&t=1s).
- Read this [short introduction to Dask](https://docs.dask.org/en/stable/10-minutes-to-dask.html). I recommend picking *either* Pandas DataFrames or NumPy arrays as the example you want to follow while reading. 
- Read this [explanation of task graphs](https://docs.dask.org/en/latest/graphs.html) provided by Dask.